# 31 · Ookla: public S3 → Bronze (global tiles)

For each quarter and connection type (one Parquet file each):

1. Skip it if `ingestion_versions` already records it as complete (so a run that stops
   half-way, e.g. on a quota, resumes where it left off).
2. Download it into the UC Volume.
3. Read it, check the columns, and derive tile centroids if an older file lacks them.
4. Overwrite exactly its partition of `bronze_ookla_tiles` (`replaceWhere`), so a re-run
   replaces rather than duplicates.
5. Reconcile: rows in the partition must equal rows in the file.
6. Record it as complete, then delete the raw file. The public, immutable S3 bucket is the
   archive; keeping a second copy would double storage (ADR 0003).

Speed test data © Ookla, 2019–present, CC BY-NC-SA 4.0.

In [0]:
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "digital_divide_dev")
dbutils.widgets.text("latest_quarters", "8")
dbutils.widgets.text("types", "fixed,mobile")
dbutils.widgets.text("delete_raw", "true")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
LATEST_QUARTERS = int(dbutils.widgets.get("latest_quarters"))
TYPES = [t.strip() for t in dbutils.widgets.get("types").split(",") if t.strip()]
DELETE_RAW = dbutils.widgets.get("delete_raw").strip().lower() == "true"

T = f"{CATALOG}.{SCHEMA}"
BRONZE = f"{T}.bronze_ookla_tiles"
INGESTION_VERSIONS = f"{T}.ingestion_versions"
VOLUME_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/landing/ookla"
S3 = "https://ookla-open-data.s3.amazonaws.com/parquet/performance"
MONTH = {1: "01", 2: "04", 3: "07", 4: "10"}

## 1 · Which quarters?

In [0]:
import os
import time
from datetime import date

import requests
from pyspark.sql import functions as F


def file_name(conn_type: str, year: int, quarter: int) -> str:
    return f"{year}-{MONTH[quarter]}-01_performance_{conn_type}_tiles.parquet"


def url(conn_type: str, year: int, quarter: int) -> str:
    return (f"{S3}/type={conn_type}/year={year}/quarter={quarter}/"
            f"{file_name(conn_type, year, quarter)}")


def previous(year: int, quarter: int) -> tuple[int, int]:
    return (year, quarter - 1) if quarter > 1 else (year - 1, 4)


# Walk back from the current quarter to the newest one Ookla has published (fixed is the reference).
today = date.today()
y, q = today.year, (today.month - 1) // 3 + 1
for _ in range(8):
    if requests.head(url("fixed", y, q), timeout=30).ok:
        break
    y, q = previous(y, q)
else:
    raise RuntimeError("No published Ookla quarter found in the last two years")

QUARTERS = []
for _ in range(LATEST_QUARTERS):
    QUARTERS.append((y, q))
    y, q = previous(y, q)
QUARTERS.reverse()
print(f"Quarters: {QUARTERS[0][0]} Q{QUARTERS[0][1]} … {QUARTERS[-1][0]} Q{QUARTERS[-1][1]} "
      f"({len(QUARTERS)}) × types {TYPES} = {len(QUARTERS) * len(TYPES)} files")

Quarters: 2024 Q4 … 2026 Q3 (8) × types ['fixed', 'mobile'] = 16 files


## 2 · Tables

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {BRONZE} (
        quadkey STRING, tile STRING, tile_x DOUBLE, tile_y DOUBLE,
        avg_d_kbps BIGINT, avg_u_kbps BIGINT, avg_lat_ms BIGINT,
        avg_lat_down_ms DOUBLE, avg_lat_up_ms DOUBLE, tests BIGINT, devices BIGINT,
        conn_type STRING, year INT, quarter INT, period_start DATE,
        centroid_derived BOOLEAN, _source_url STRING, _loaded_at TIMESTAMP
    )
    PARTITIONED BY (conn_type, year, quarter)
    COMMENT 'Ookla global speed-test tiles (zoom 16), one partition per connection type and quarter'
""")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {INGESTION_VERSIONS} (
        source STRING NOT NULL, source_updated STRING NOT NULL, indicators ARRAY<STRING>,
        countries BOOLEAN, files INT, `rows` BIGINT, completed_at TIMESTAMP
    ) COMMENT 'Releases fully landed in Bronze and reconciled'
""")

SOURCE_COLUMNS = {
    "quadkey": "string", "tile": "string", "tile_x": "double", "tile_y": "double",
    "avg_d_kbps": "bigint", "avg_u_kbps": "bigint", "avg_lat_ms": "bigint",
    "avg_lat_down_ms": "double", "avg_lat_up_ms": "double", "tests": "bigint", "devices": "bigint",
}
REQUIRED = {"quadkey", "tile", "avg_d_kbps", "avg_u_kbps", "avg_lat_ms", "tests", "devices"}

## 3 · Load each file

In [0]:

def completed_rows(source: str, period: str):
    r = (spark.table(INGESTION_VERSIONS)
         .where((F.col("source") == source) & (F.col("source_updated") == period))
         .select("rows").first())
    return r["rows"] if r else None


def record_complete(source: str, period: str, rows: int) -> None:
    spark.createDataFrame(
        [(source, period, [], False, 1, rows)],
        "source STRING, source_updated STRING, indicators ARRAY<STRING>, countries BOOLEAN, "
        "files INT, `rows` BIGINT",
    ).createOrReplaceTempView("completed_file")
    spark.sql(f"""
        MERGE INTO {INGESTION_VERSIONS} t USING completed_file s
        ON t.source = s.source AND t.source_updated = s.source_updated
        WHEN MATCHED THEN UPDATE SET t.`rows` = s.`rows`, t.files = s.files,
                                     t.completed_at = current_timestamp()
        WHEN NOT MATCHED THEN INSERT (source, source_updated, indicators, countries, files, `rows`,
                                      completed_at)
                              VALUES (s.source, s.source_updated, s.indicators, s.countries,
                                      s.files, s.`rows`, current_timestamp())
    """)


results = []
for conn_type in TYPES:
    for year, quarter in QUARTERS:
        source, period = f"ookla_{conn_type}", f"{year}-{MONTH[quarter]}-01"
        partition = f"conn_type = '{conn_type}' AND year = {year} AND quarter = {quarter}"
        label = f"{conn_type} {year} Q{quarter}"

        recorded = completed_rows(source, period)
        if recorded is not None and spark.table(BRONZE).where(partition).count() == recorded:
            results.append((label, "skipped (complete)", recorded, 0.0))
            continue

        start = time.time()
        src_url = url(conn_type, year, quarter)
        target = f"{VOLUME_ROOT}/type={conn_type}/year={year}/quarter={quarter}/" \
                 f"{file_name(conn_type, year, quarter)}"
        size = int(requests.head(src_url, timeout=30).headers["Content-Length"])
        if not (os.path.exists(target) and os.path.getsize(target) == size):
            os.makedirs(os.path.dirname(target), exist_ok=True)
            with requests.get(src_url, stream=True, timeout=120) as r, open(target, "wb") as out:
                r.raise_for_status()
                for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
                    out.write(chunk)

        raw = spark.read.parquet(target)
        missing = REQUIRED - set(raw.columns)
        assert not missing, f"{label}: required columns missing: {sorted(missing)}"

        # Older files may lack the centroid or the split latency columns: derive or fill.
        derived = not {"tile_x", "tile_y"} <= set(raw.columns)
        if derived:
            raw = raw.withColumn("tile_x", F.expr("st_x(st_centroid(st_geomfromtext(tile)))")) \
                     .withColumn("tile_y", F.expr("st_y(st_centroid(st_geomfromtext(tile)))"))
        for col, dtype in SOURCE_COLUMNS.items():
            if col not in raw.columns:
                raw = raw.withColumn(col, F.lit(None).cast(dtype))

        source_rows = raw.count()
        bronze_df = raw.select(
            *[F.col(c).cast(t).alias(c) for c, t in SOURCE_COLUMNS.items()],
            F.lit(conn_type).alias("conn_type"),
            F.lit(year).cast("int").alias("year"),
            F.lit(quarter).cast("int").alias("quarter"),
            F.to_date(F.lit(period)).alias("period_start"),
            F.lit(derived).alias("centroid_derived"),
            F.lit(src_url).alias("_source_url"),
            F.current_timestamp().alias("_loaded_at"),
        )
        (bronze_df.write.format("delta").mode("overwrite")
            .option("replaceWhere", partition).saveAsTable(BRONZE))

        written = spark.table(BRONZE).where(partition).count()
        assert written == source_rows, f"{label}: {written} rows written, {source_rows} in the file"
        record_complete(source, period, written)
        if DELETE_RAW:
            os.remove(target)
        results.append((label, "loaded" + (", centroid derived" if derived else ""),
                        written, round(time.time() - start, 1)))
        print(f"{label}: {written:,} rows in {results[-1][3]} s")

fixed 2024 Q4: 6,561,086 rows in 43.2 s
fixed 2025 Q1: 6,364,159 rows in 30.9 s
fixed 2025 Q2: 6,797,378 rows in 29.0 s
fixed 2025 Q3: 6,936,126 rows in 30.3 s
fixed 2025 Q4: 6,618,380 rows in 29.4 s
fixed 2026 Q1: 6,312,198 rows in 28.1 s
fixed 2026 Q2: 6,435,895 rows in 26.8 s
fixed 2026 Q3: 6,640,362 rows in 20.0 s
mobile 2024 Q4: 3,551,267 rows in 21.9 s
mobile 2025 Q1: 3,388,115 rows in 20.8 s
mobile 2025 Q2: 3,514,872 rows in 21.5 s
mobile 2025 Q3: 3,409,380 rows in 21.7 s
mobile 2025 Q4: 3,311,092 rows in 20.0 s
mobile 2026 Q1: 3,186,269 rows in 20.0 s
mobile 2026 Q2: 3,381,216 rows in 18.8 s
mobile 2026 Q3: 3,477,926 rows in 19.7 s


## 4 · Summary and storage

In [0]:
display(spark.createDataFrame(results, "file STRING, status STRING, rows BIGINT, seconds DOUBLE"))

detail = spark.sql(f"DESCRIBE DETAIL {BRONZE}").first()
total_rows = spark.table(BRONZE).count()
loaded = sum(r[2] for r in results if r[1].startswith("loaded"))
print(f"Loaded this run: {loaded:,} rows | Bronze total: {total_rows:,} rows | "
      f"Delta size: {detail['sizeInBytes'] / 1e9:.2f} GB in {detail['numFiles']} files")
display(spark.sql(f"""
    SELECT conn_type, year, quarter, count(*) AS tiles, sum(tests) AS tests
    FROM {BRONZE} GROUP BY ALL ORDER BY conn_type, year, quarter
"""))

file,status,rows,seconds
fixed 2024 Q4,loaded,6561086,43.2
fixed 2025 Q1,loaded,6364159,30.9
fixed 2025 Q2,loaded,6797378,29.0
fixed 2025 Q3,loaded,6936126,30.3
fixed 2025 Q4,loaded,6618380,29.4
fixed 2026 Q1,loaded,6312198,28.1
fixed 2026 Q2,loaded,6435895,26.8
fixed 2026 Q3,loaded,6640362,20.0
mobile 2024 Q4,loaded,3551267,21.9
mobile 2025 Q1,loaded,3388115,20.8


Loaded this run: 79,885,721 rows | Bronze total: 79,885,721 rows | Delta size: 2.74 GB in 16 files


conn_type,year,quarter,tiles,tests
fixed,2024,4,6561086,104326033
fixed,2025,1,6364159,97329055
fixed,2025,2,6797378,108922896
fixed,2025,3,6936126,111478076
fixed,2025,4,6618380,111445449
fixed,2026,1,6312198,100973933
fixed,2026,2,6435895,99815901
fixed,2026,3,6640362,104953622
mobile,2024,4,3551267,20813531
mobile,2025,1,3388115,19156257
